# 授業動画自動生成システム（HTML/PDF/SVGアニメーション対応版）

MarpのHTML/PDFスライドと口述原稿から、音声付き動画を自動生成します。
SVGアニメーションにも対応しています。

## 機能
- **HTMLスライドから動画生成**
- **PDFスライドから動画生成（新機能）**
- SVGアニメーションの自動検出と録画
- 日本語フォント対応
- 音声合成（Gemini API）
- 検証モード（音声なしで動画のみ生成）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd "./drive/MyDrive/path/to/your/lecture_folder"


In [ ]:
ls

In [ ]:
prompt = """Speak as an experienced teacher. Your delivery should be clear and articulate. Use engaging intonation and vocal emphasis to capture and maintain student interest:
"""

In [ ]:
prompt2 = """TTS the following conversation between Expert and Novice:
Make Expert sound like an experienced teacher. Expert's delivery should be clear and articulate, using engaging intonation and vocal emphasis to capture and maintain student interest.
Make Novice sound like a curious and energetic young boy.
"""

In [ ]:
prompt3 = """TTS the following conversation between Peer A and Peer B, simulating a peer-to-peer learning session:
Make Peer A sound like a knowledgeable and articulate student. Their delivery should be clear, confident, and engaging, carefully explaining concepts as if to a classmate.
Make Peer B sound like a curious and energetic classmate. They should sound actively engaged, asking thoughtful questions and offering different perspectives to deepen the discussion.
"""

In [ ]:
prompt4 = """TTS the following conversation between an Expert and a Facilitator, simulating a structured educational discussion:
Make the Expert (Subject-Matter Expert) sound like a seasoned professional and authority in their field. Their delivery should be clear, precise, and insightful, conveying deep knowledge with confidence.
Make the Facilitator (Pedagogical Facilitator) sound like an engaging host and educator. They should sound supportive and curious, guiding the conversation, asking clarifying questions from a learner's perspective, and summarizing key takeaways.
"""

In [ ]:
# プロンプトの定義
prompts = {
    1: prompt,   # 単一話者用
    2: prompt2,  # Expert-Novice用
    3: prompt3,  # Peer用
    4: prompt4   # Expert-Facilitator用
}

# 準備

## 1. 必要なライブラリのインストール

In [ ]:
# 1. 必要なライブラリをインストールする (Chromeの実行に必要)
!apt-get update
!apt-get install -y libglib2.0-0 \
    libnss3 \
    libgconf-2-4 \
    libfontconfig1 \
    libx11-xcb1 \
    libxcomposite1 \
    libxcursor1 \
    libxdamage1 \
    libxext6 \
    libxfixes3 \
    libxi6 \
    libxrandr2 \
    libxrender1 \
    libxss1 \
    libxtst6 \
    ca-certificates \
    fonts-liberation \
    libappindicator1 \
    libasound2 \
    libatk1.0-0 \
    libcairo2 \
    libcups2 \
    libgbm1 \
    libgtk-3-0 \
    libpango-1.0-0 \
    xdg-utils \
    wget

# 2. Google Chromeの最新安定版をダウンロードしてインストールする
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb

# 3. Pythonライブラリをインストールする
!pip install selenium webdriver-manager
!pip install pdf2image


In [ ]:
# ============================================================================
# 1. 必要なライブラリのインストール
# ============================================================================

# 基本パッケージ
!pip install pdf2image moviepy
!pip install selenium beautifulsoup4 lxml opencv-python webdriver-manager
!pip install cairosvg  # SVGレンダリング改善

# システムパッケージ
!apt-get update -qq
!apt-get install -y poppler-utils chromium-browser chromium-chromedriver

# 日本語フォントのインストール
!apt-get install -y fonts-noto-cjk fonts-ipafont-gothic fonts-ipafont-mincho
!fc-cache -fv

print("\n✅ インストール完了！")

In [ ]:
# ==========================================
# 依存関係の修復と再インストール
# ==========================================

# 1. パッケージリストの更新
!apt-get update -qq

# 2. 壊れた依存関係の修復
!apt --fix-broken install -y

# 3. 必要なライブラリのインストール (libvulkan1はChrome用, poppler-utilsはPDF用)
!apt-get install -y libvulkan1 poppler-utils

# 4. poppler-utilsがインストールされたか確認
!pdfinfo -v

print("\n✅ 修復とインストールが完了しました。再度メイン処理を実行してください。")

In [ ]:
!apt-get update -qq
!apt-get install -y ffmpeg
!pip install imageio imageio-ffmpeg

## 2. ライブラリのインポートと設定

In [ ]:
# ============================================================================
# 2. ライブラリのインポートと設定
# ============================================================================

import os
import re
import json
import shutil
from pathlib import Path
from typing import List, Tuple, Dict, Set, Optional
import tempfile
import time
import sys

# Google Colab用ファイルアップロード
from google.colab import files

# PDF処理（オプション）
from pdf2image import convert_from_path

# HTML解析
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

# Webブラウザ制御
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 動画処理
from moviepy.editor import ImageClip, AudioFileClip, VideoFileClip, concatenate_videoclips
import cv2
import numpy as np
from PIL import Image
import io

# Google AI関連
from google import genai
from google.genai import types
import wave
from pydub import AudioSegment

import warnings
warnings.filterwarnings('ignore')

## 3. 基本設定とユーティリティ関数

In [ ]:
# ============================================================================
# 3. 基本設定とユーティリティ関数
# ============================================================================

# 出力ディレクトリ設定
OUTPUT_DIR = "slide_output"

def setup_output_directory():
    """出力ディレクトリを作成"""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_DIR}/audio", exist_ok=True)
    print(f"✅ 出力ディレクトリ準備完了: {OUTPUT_DIR}")

def setup_gemini_client():
    """Gemini AI クライアントをセットアップ"""
    import getpass
    print("\n=== Gemini API設定 ===")
    api_key = getpass.getpass("Gemini APIキーを入力してください: ")
    return genai.Client(api_key=api_key)

def setup_chrome_driver():
    """Chrome WebDriverのセットアップ（日本語フォント対応）"""
    chrome_options = Options()
    chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1280,720')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--lang=ja-JP')  # 日本語対応
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

def trigger_svg_animations(driver):
    """SVGアニメーションを強制的に開始"""
    js_code = """
    // すべてのSVG要素を取得
    const svgs = document.querySelectorAll('svg');

    svgs.forEach(svg => {
        // SVG内のアニメーション要素を取得
        const animations = svg.querySelectorAll('animate, animateTransform, animateMotion');

        animations.forEach(anim => {
            // アニメーションをリスタート
            anim.beginElement();
        });
    });

    // インラインSVGも含めて再度チェック
    const allAnimations = document.querySelectorAll('animate, animateTransform, animateMotion');
    allAnimations.forEach(anim => {
        try {
            anim.beginElement();
        } catch (e) {
            console.log('Animation start failed:', e);
        }
    });
    """
    driver.execute_script(js_code)

## 4. ファイルアップロードとリソース管理

In [ ]:
# ============================================================================
# 4. ファイルアップロードとリソース管理（PDF対応版）
# ============================================================================

def upload_main_files():
    """
    メインファイル（HTML/PDFとMarkdown）をアップロード

    Returns:
        tuple: (SLIDE_PATH, MD_PATH, FILE_TYPE) のファイルパスとタイプ
        FILE_TYPE は 'html' または 'pdf'
    """
    print("=== ファイルアップロード ===")
    print("以下のファイルをアップロードしてください：")
    print("1. スライド.html または スライド.pdf (Marpのファイル)")
    print("2. スライド_script.md (口述原稿付きMarkdownファイル)")
    print()

    uploaded = files.upload()

    slide_path = None
    md_path = None
    file_type = None

    for filename in uploaded.keys():
        if filename.endswith('.html'):
            slide_path = filename
            file_type = 'html'
            print(f"✅ HTMLファイル: {filename}")
        elif filename.endswith('.pdf'):
            slide_path = filename
            file_type = 'pdf'
            print(f"✅ PDFファイル: {filename}")
        elif filename.endswith('.md'):
            md_path = filename
            print(f"✅ Markdownファイル: {filename}")
        else:
            print(f"⚠️  認識されないファイル: {filename}")

    if not slide_path:
        raise FileNotFoundError("HTMLまたはPDFファイルがアップロードされていません")
    if not md_path:
        raise FileNotFoundError("Markdownファイルがアップロードされていません")

    print(f"\n✅ ファイルアップロード完了")
    print(f"📄 スライドタイプ: {file_type.upper()}")
    return slide_path, md_path, file_type

def analyze_html_resources(html_path: str) -> Dict[str, Set[str]]:
    """
    HTMLファイルを解析して必要なリソースファイルを特定
    """
    resources = {
        'images': set(),
        'svgs': set(),
        'others': set()
    }

    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    # img タグから画像ファイルを抽出
    for img in soup.find_all('img'):
        src = img.get('src', '')
        if src and not src.startswith(('http://', 'https://', 'data:')):
            if src.endswith('.svg'):
                resources['svgs'].add(src)
            elif src.endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp')):
                resources['images'].add(src)
            else:
                resources['others'].add(src)

    # object, embed タグも確認
    for tag in soup.find_all(['object', 'embed']):
        src = tag.get('data') or tag.get('src')
        if src and src.endswith('.svg') and not src.startswith(('http://', 'https://')):
            resources['svgs'].add(src)

    return resources

def upload_required_resources(resources: Dict[str, Set[str]]) -> Dict[str, str]:
    """
    必要なリソースファイルのアップロードを促す
    """
    all_files = resources['svgs'] | resources['images'] | resources['others']

    if not all_files:
        print("追加のリソースファイルは必要ありません。")
        return {}

    print("\n=== 追加リソースファイルのアップロード ===")
    print("以下のファイルをアップロードしてください：")

    for category, file_list in resources.items():
        if file_list:
            print(f"\n{category.upper()}:")
            for file in sorted(file_list):
                print(f"  - {file}")

    print("\n※ ディレクトリ構造を保持する必要がある場合は、")
    print("   ファイル名を 'fig__図1_エネルギー最小化.svg' のように")
    print("   '__' でディレクトリを区切って命名してください。")

    uploaded = files.upload()

    # アップロードされたファイルを整理
    file_mapping = {}
    temp_dir = Path("uploaded_resources")
    temp_dir.mkdir(exist_ok=True)

    for filename, content in uploaded.items():
        # ファイル名に '__' が含まれている場合、ディレクトリ構造を復元
        if '__' in filename:
            parts = filename.split('__')
            subdir = temp_dir / parts[0]
            subdir.mkdir(exist_ok=True)
            file_path = subdir / parts[1]
            original_path = f"{parts[0]}/{parts[1]}"
        else:
            file_path = temp_dir / filename
            original_path = filename

        # ファイルを保存
        with open(file_path, 'wb') as f:
            f.write(content)

        file_mapping[original_path] = str(file_path)
        print(f"✅ {original_path} → {file_path}")

    return file_mapping


In [ ]:
# ============================================================================
# 4.5. PDF処理関数（新機能）
# ============================================================================

# 動画の解像度設定
VIDEO_WIDTH = 1920
VIDEO_HEIGHT = 1080

def extract_slides_from_pdf(pdf_path: str, output_dir: str, dpi: int = 200) -> list:
    """
    PDFファイルからスライド画像を抽出

    Args:
        pdf_path: PDFファイルのパス
        output_dir: 出力ディレクトリ
        dpi: 画像の解像度

    Returns:
        抽出された画像ファイルのパスのリスト
    """
    print(f"\n📄 PDFからスライド画像を抽出: {pdf_path}")

    # 出力ディレクトリの作成
    pdf_slides_dir = os.path.join(output_dir, "pdf_slides")
    os.makedirs(pdf_slides_dir, exist_ok=True)

    # PDFを画像に変換
    print(f"PDFを画像に変換中（DPI: {dpi}）...")
    images = convert_from_path(pdf_path, dpi=dpi)

    image_paths = []
    for i, image in enumerate(images):
        # 1920x1080にリサイズ
        image = image.resize((VIDEO_WIDTH, VIDEO_HEIGHT), Image.Resampling.LANCZOS)

        # 画像を保存
        image_path = os.path.join(pdf_slides_dir, f"slide_{i:03d}.png")
        image.save(image_path, "PNG")
        image_paths.append(image_path)

        print(f"  ✅ スライド {i+1}/{len(images)} を抽出")

    print(f"\n📊 合計 {len(image_paths)} 枚のスライドを抽出しました")
    return image_paths


def pdf_slides_to_video_clips(image_paths: list) -> tuple:
    """
    PDFから抽出した画像をmovie_pathsとvideo_clipsの形式に変換

    Args:
        image_paths: 画像パスのリスト

    Returns:
        tuple: (media_paths, video_clips)
               - media_paths: 画像パスのリスト
               - video_clips: すべてNone（静止画のため）
    """
    media_paths = image_paths
    video_clips = [None] * len(image_paths)  # 静止画なのでNone

    return media_paths, video_clips

## 5. 口述原稿抽出関数

In [ ]:
import re
from typing import List

# ============================================================================
# 5. 口述原稿抽出関数（修正版）
# ============================================================================

def extract_speech_scripts(md_path: str) -> List[str]:
    """
    Markdownファイルからスライドごとに口述原稿を抽出する。
    口述原稿がないスライドは空文字をリストに追加する。

    Args:
        md_path: Markdownファイルのパス

    Returns:
        口述原稿のリスト（スライド順）。原稿がない場合は空文字。
    """
    try:
        with open(md_path, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"エラー: ファイルが見つかりません。パスを確認してください: {md_path}")
        return []

    # スライド区切り文字 '---' で全体を分割
    # re.splitを使うことで、前後の空白や改行に柔軟に対応
    slides = re.split(r'\n---\n', content.strip())
    comment_pattern = r'<!--\s*(.*?)\s*-->'

    scripts = []
    # 各スライドをループ処理
    for slide_content in slides[1:]:
        #print(slide_content)
        # スライド内のHTMLコメントを検索（re.DOTALLで改行を含む検索を可能に）
        comments = re.findall(comment_pattern, slide_content, re.DOTALL)
        print(comments)
        #comment_match = re.search(r'', slide_content, re.DOTALL)

        found_script_text = ""  # デフォルト値として空文字を設定
        comment = comments[-1]
        if '口述原稿：' in comment or '口述原稿:' in comment:
            lines = comment.split('\n')
            script_started = False
            script_lines = []

            for line in lines:
                if '口述原稿：' in line or '口述原稿:' in line:
                    script_started = True
                    after_colon = re.split(r'口述原稿[：:]', line, 1)
                    if len(after_colon) > 1 and after_colon[1].strip():
                        script_lines.append(after_colon[1].strip())
                elif script_started and line.strip():
                    script_lines.append(line.strip())

            if script_lines:
                script_text = '\n'.join(script_lines)
                script_text = re.sub(r'\*\*', '', script_text)
                script_text = re.sub(r'「', '', script_text)
                script_text = re.sub(r'」', '', script_text)
                scripts.append(script_text.strip())
        else:
          # 見つかった原稿（または空文字）をリストに追加
          scripts.append("")

    return scripts

## 6. アニメーション検出関数（改良版）

In [ ]:
# ============================================================================
# 6. アニメーション検出関数（改良版）
# ============================================================================

def check_svg_animation(svg_path: str) -> Tuple[bool, float]:
    """
    SVGファイルにアニメーションが含まれているかチェック（改良版）

    Returns:
        (アニメーションの有無, 最大継続時間)
    """
    try:
        with open(svg_path, 'r', encoding='utf-8') as f:
            svg_content = f.read()

        # アニメーション要素のパターン
        animation_patterns = [
            r'<animate\s',
            r'<animateTransform\s',
            r'<animateMotion\s',
            r'<animateColor\s',
            r'<set\s',
            r'@keyframes',  # CSSアニメーション
            r'animation:',   # CSSアニメーションプロパティ
            r'transition:'   # CSSトランジション
        ]

        has_animation = False
        max_duration = 0.0

        for pattern in animation_patterns:
            if re.search(pattern, svg_content, re.IGNORECASE):
                has_animation = True

                # 継続時間を抽出
                dur_patterns = [
                    r'dur=["\']?(\d+(?:\.\d+)?)(s|ms)?["\']?',
                    r'animation-duration:\s*(\d+(?:\.\d+)?)(s|ms)',
                    r'transition:\s*\w+\s+(\d+(?:\.\d+)?)(s|ms)'
                ]

                for dur_pattern in dur_patterns:
                    matches = re.findall(dur_pattern, svg_content, re.IGNORECASE)
                    for match in matches:
                        value = float(match[0])
                        unit = match[1] if len(match) > 1 else 's'

                        if unit == 'ms':
                            value /= 1000

                        max_duration = max(max_duration, value)

        # repeatCount="indefinite" の場合
        if 'repeatCount="indefinite"' in svg_content or 'repeatCount=\'indefinite\'' in svg_content:
            max_duration = max(max_duration, 3.0)

        # アニメーションがある場合、最低でも2秒は確保
        if has_animation and max_duration < 2.0:
            max_duration = 2.0

        return has_animation, max_duration

    except Exception as e:
        print(f"SVGファイル解析エラー ({svg_path}): {e}")
        return False, 0.0

def detect_animation_slides(html_path: str, file_mapping: Dict[str, str] = None) -> Dict[int, float]:
    """
    HTMLとSVGファイルからアニメーションスライドを検出
    """
    animation_info = {}

    with open(html_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # インラインアニメーションの検出
    slides = content.split('<section')

    for i, slide in enumerate(slides[1:], 0):
        # インラインSVGアニメーション
        if any(tag in slide for tag in ['<animate', 'animateTransform', 'animateMotion', '@keyframes', 'animation:']):
            dur_matches = re.findall(r'dur="(\d+(?:\.\d+)?)s?"', slide)
            if dur_matches:
                max_duration = max(float(d.replace('s', '')) for d in dur_matches)
                animation_info[i] = max_duration + 0.5
            else:
                animation_info[i] = 3.0

        # 外部SVGファイルのアニメーション
        if file_mapping:
            svg_refs = re.findall(r'src=["\']([^"\'\']+\.svg)["\']', slide)

            for svg_ref in svg_refs:
                actual_path = None
                for pattern in [svg_ref, './' + svg_ref, svg_ref.lstrip('./')]:
                    if pattern in file_mapping:
                        actual_path = file_mapping[pattern]
                        break

                if actual_path and os.path.exists(actual_path):
                    has_animation, duration = check_svg_animation(actual_path)
                    if has_animation:
                        if i in animation_info:
                            animation_info[i] = max(animation_info[i], duration + 0.5)
                        else:
                            animation_info[i] = duration + 0.5

    return animation_info

## 7. HTML → 画像/動画変換関数（改良版）

In [ ]:
# ============================================================================
# 7. HTML → 画像/動画変換関数（改良版）
# ============================================================================

def create_temp_html_with_resources(html_path: str, file_mapping: Dict[str, str]) -> str:
    """
    リソースパスを更新した一時的なHTMLファイルを作成
    """
    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    # リソースパスを絶対パスに置換
    for original_path, actual_path in file_mapping.items():
        abs_path = os.path.abspath(actual_path)
        file_url = f"file://{abs_path}"

        patterns = [
            f'src="{original_path}"',
            f"src='{original_path}'",
            f'data="{original_path}"',
            f"data='{original_path}'",
            f'url("{original_path}")',
            f"url('{original_path}')",
            f'url({original_path})'
        ]

        for pattern in patterns:
            html_content = html_content.replace(pattern, pattern.replace(original_path, file_url))

    temp_html = "temp_presentation.html"
    with open(temp_html, 'w', encoding='utf-8') as f:
        f.write(html_content)

    return temp_html

def capture_slide_with_animation(driver, animation_duration=5.0):
    """
    アニメーション付きスライドをVideoClipとして返す（ファイル生成しない）
    """
    from moviepy.editor import ImageClip, concatenate_videoclips
    from PIL import Image
    import io

    # アニメーションを強制的に開始
    trigger_svg_animations(driver)
    time.sleep(0.5)

    fps = 30
    frame_duration = 1.0 / fps
    frame_count = int(animation_duration * fps)

    # フレームをMoviePyクリップとして収集
    clips = []
    start_time = time.time()

    for i in range(frame_count):
        # スクリーンショットを取得
        screenshot = driver.get_screenshot_as_png()
        img = Image.open(io.BytesIO(screenshot))
        frame = np.array(img)

        # フレームをクリップとして追加
        clip = ImageClip(frame).set_duration(frame_duration)
        clips.append(clip)

        # タイミング調整
        expected_time = start_time + (i + 1) / fps
        actual_time = time.time()
        if actual_time < expected_time:
            time.sleep(expected_time - actual_time)

    # すべてのフレームを結合して動画クリップを作成
    video_clip = concatenate_videoclips(clips, method="compose")

    # 個別のクリップをクリーンアップ
    for clip in clips:
        clip.close()

    return video_clip

def setup_chrome_driver():
    """Chrome WebDriverをセットアップ（数式レンダリング対応版）"""
    from selenium import webdriver

    options = webdriver.ChromeOptions()
    options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    # 高解像度レンダリング
    options.add_argument('--force-device-scale-factor=2')
    options.add_argument('--window-size=1920,1080')

    # フォントレンダリングを改善
    options.add_argument('--font-render-hinting=none')
    options.add_argument('--disable-font-subpixel-positioning')

    driver = webdriver.Chrome(options=options)
    return driver

def wait_for_fonts_and_katex(driver, timeout=15):
    """
    KaTeXのレンダリングとフォントの読み込み完了を待機
    """
    # 1. すべてのWebフォントが読み込まれるまで待つ
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.fonts.ready.then(() => true)")
    )

    # 2. KaTeXの数式がすべてレンダリングされるまで待つ
    driver.execute_script("""
        return new Promise((resolve) => {
            // KaTeXの数式要素が存在するか確認
            const katexElements = document.querySelectorAll('.katex, .katex-html');

            if (katexElements.length === 0) {
                // 数式がなければすぐに完了
                resolve(true);
                return;
            }

            // すべての数式要素が完全にレンダリングされるまで待つ
            const checkRendering = () => {
                const allRendered = Array.from(katexElements).every(el => {
                    return el.offsetHeight > 0 && el.offsetWidth > 0;
                });

                if (allRendered) {
                    resolve(true);
                } else {
                    setTimeout(checkRendering, 100);
                }
            };

            checkRendering();
        });
    """)

    # 3. 画像やSVGの読み込み完了を待つ
    driver.execute_script("""
        return Promise.all(
            Array.from(document.images)
                .filter(img => !img.complete)
                .map(img => new Promise(resolve => {
                    img.onload = img.onerror = resolve;
                }))
        );
    """)

    # 4. 念のため追加の待機時間
    time.sleep(1)

def html_to_slides(html_path: str, output_dir: str,
                  animation_slides: Dict[int, float] = None) -> Tuple[List[str], List]:
    """
    HTMLファイルから各スライドをキャプチャ（数式対応版）
    """
    if animation_slides is None:
        animation_slides = {}

    print("HTMLからスライドをキャプチャ中...")

    driver = setup_chrome_driver()
    output_paths = []
    video_clips = []

    try:
        # HTMLファイルを開く
        abs_html_path = os.path.abspath(html_path)
        driver.get(f"file://{abs_html_path}")

        # Marpのスライドが読み込まれるまで待機
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".bespoke-marp-parent"))
        )

        # **重要: フォントとKaTeXの読み込み完了を待つ**
        print("  数式とフォントの読み込みを待機中...")
        wait_for_fonts_and_katex(driver)
        print("  読み込み完了")

        # 追加のフォント設定（オプション：日本語フォント用）
        font_css = """
        /* ベースのフォント設定 */
        * {
            font-family: 'Noto Sans CJK JP', 'Noto Sans JP', 'IPAGothic',
                         'Hiragino Kaku Gothic ProN', 'Yu Gothic',
                         'Meiryo',
                         'Segoe UI Emoji', 'Apple Color Emoji', 'Noto Color Emoji', /* 絵文字用フォントを追加 */
                         sans-serif !important;
        }

        /* KaTeX数式フォントの設定 */
        .katex, .katex * {
            font-family: 'KaTeX_Main', 'KaTeX_Math', 'KaTeX_AMS',
                         'Noto Sans CJK JP', 'Noto Sans JP', 'IPAGothic',
                         'Hiragino Kaku Gothic ProN', 'Yu Gothic',
                         'Meiryo',
                         'Segoe UI Emoji', 'Apple Color Emoji', 'Noto Color Emoji', /* 数式内でも絵文字が出せるように追加 */
                         sans-serif !important;
        }
        """
        driver.execute_script(f"""
            const style = document.createElement('style');
            style.textContent = `{font_css}`;
            document.head.appendChild(style);
        """)

        # スライド数を取得
        slides = driver.find_elements(By.CSS_SELECTOR, "svg.bespoke-marp-slide")
        slide_count = len(slides)
        print(f"スライド数: {slide_count}")

        for i in range(slide_count):
            print(f"  スライド {i+1} を処理中...")

            # スライドに移動
            if i > 0:
                driver.execute_script("document.dispatchEvent(new KeyboardEvent('keydown', {key: 'ArrowRight'}))")
                time.sleep(0.8)  # スライド遷移アニメーションを待つ

                # スライド遷移後も数式の再レンダリングを待つ
                wait_for_fonts_and_katex(driver, timeout=5)

            # 現在のスライドでSVGアニメーションをトリガー
            trigger_svg_animations(driver)

            # アニメーションがあるスライドかチェック
            if i in animation_slides:
                time.sleep(0.5)
                video_clip = capture_slide_with_animation(driver, animation_slides[i])
                video_clips.append(video_clip)
                output_paths.append(None)
                print(f"    アニメーション録画完了（VideoClipオブジェクト）")
            else:
                time.sleep(0.5)
                image_path = f"{output_dir}/images/slide_{i+1:02d}.png"
                driver.save_screenshot(image_path)
                output_paths.append(image_path)
                video_clips.append(None)
                print(f"    画像保存完了: {image_path}")

    finally:
        driver.quit()

    print(f"完了: {len(output_paths)}個のファイル/オブジェクトを生成")
    return output_paths, video_clips


## 8. 音声合成関数

In [ ]:
# ============================================================================
# 8. 音声合成関数
# ============================================================================

def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)


def text_to_speech(
    client: genai,
    text: str,
    output_path: str,
    prompt: str,
    speaker_config: dict = None
) -> str:
    """
    Gemini APIを使用してテキストを音声に変換

    Args:
        client: Gemini APIクライアント
        text: 変換するテキスト
        output_path: 出力ファイルパス
        prompt: 使用するプロンプト
        speaker_config: 話者設定 (Noneの場合は単一話者)
            例: {
                'Expert': 'Enceladus',
                'Novice': 'Autonoe'
            }
    """
    try:
        # 音声設定の構築
        if speaker_config is None:
            # 単一話者モード
            speech_config = types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Enceladus'
                    )
                )
            )
        else:
            # 複数話者モード
            speaker_voice_configs = [
                types.SpeakerVoiceConfig(
                    speaker=speaker_name,
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name=voice_name
                        )
                    )
                )
                for speaker_name, voice_name in speaker_config.items()
            ]
            speech_config = types.SpeechConfig(
                multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                    speaker_voice_configs=speaker_voice_configs
                )
            )

        # 音声生成
        response = client.models.generate_content(
            model="gemini-2.5-pro-preview-tts",
            contents=prompt + text,
            config=types.GenerateContentConfig(
                response_modalities=["AUDIO"],
                speech_config=speech_config
            )
        )

        # 音声データの取得と処理
        data = response.candidates[0].content.parts[0].inline_data.data
        wave_file('tmp.wav', data)
        sound = AudioSegment.from_file("tmp.wav")

        # 前後に無音を追加
        silence_duration = 1000
        silence = AudioSegment.silent(
            duration=silence_duration,
            frame_rate=sound.frame_rate
        )

        sound = silence + sound + silence
        sound.export(output_path, format="mp3")
        os.remove('tmp.wav')

        return output_path

    except Exception as e:
        print(f"音声合成エラー: {e}")
        return None


# 話者設定のプリセット
SPEAKER_CONFIGS = {
    'single': None,  # 単一話者
    'expert_novice': {
        'Expert': 'Enceladus',
        'Novice': 'Autonoe'
    },
    'peer': {
        'Peer A': 'Enceladus',
        'Peer B': 'Autonoe'
    },
    'expert_facilitator': {
        'Expert': 'Enceladus',
        'Facilitator': 'Autonoe'
    }
}


def detect_speech_mode(script: str) -> tuple:
    """
    スクリプトの内容から適切な話者設定とプロンプト番号を検出

    Returns:
        (speaker_config, prompt_number): 話者設定とプロンプト番号のタプル
    """
    if "Novice:" in script:
        return SPEAKER_CONFIGS['expert_novice'], 2
    elif "Peer" in script:
        return SPEAKER_CONFIGS['peer'], 3
    elif "Facilitator:" in script:
        return SPEAKER_CONFIGS['expert_facilitator'], 4
    else:
        return SPEAKER_CONFIGS['single'], 1


def generate_audio_files(
    client: genai,
    scripts: List[str],
    output_dir: str,
    prompts: dict
) -> List[str]:
    """
    口述原稿リストから音声ファイルを生成

    Args:
        client: Gemini APIクライアント
        scripts: 口述原稿のリスト
        output_dir: 出力ディレクトリ
        prompts: プロンプト辞書 {1: prompt, 2: prompt2, 3: prompt3, 4: prompt4}
    """
    audio_paths = []

    for i, script in enumerate(scripts):
        if script.strip():
            audio_path = f"{output_dir}/audio/script_{i+1:02d}.mp3"
            print(f"  スライド {i+1} の音声を生成中...")

            # 話者設定とプロンプト番号を検出
            speaker_config, prompt_number = detect_speech_mode(script)
            selected_prompt = prompts.get(prompt_number, prompts[1])

            # 音声生成
            result_path = text_to_speech(
                client,
                script,
                audio_path,
                selected_prompt,
                speaker_config
            )

            if result_path:
                audio_paths.append(result_path)
                print(f"    完了: {audio_path}")
            else:
                print(f"    失敗: スライド {i+1}")
                audio_paths.append(None)

            time.sleep(5)  # API制限を避けるため
        else:
            print(f"  スライド {i+1}: 口述原稿なし")
            audio_paths.append(None)

    print(f"完了: {len([p for p in audio_paths if p])}個の音声ファイルを生成")
    return audio_paths

## 9. 動画作成関数（高速化・最終ログ修正版

In [ ]:
# ===========================================================================
# 9. 動画作成関数（高速化・最終ログ修正版）
# ===========================================================================
import os
import shutil
from moviepy.editor import VideoFileClip, ImageClip, AudioFileClip, concatenate_videoclips
from moviepy.video.fx import all as vfx

def create_mixed_slide_video(media_paths: List[str], video_clips: List,
                           audio_paths: List[str], output_path: str,
                           default_duration: float = 20.0) -> str:
    """
    【高速化・最終ログ修正版】
    中間ファイルの生成・読込ログを抑制し、method="chain"で高速に結合する。
    """
    print("スライド動画を作成中（高速化モード）...")

    temp_dir = "temp_video_clips"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    intermediate_clip_paths = []

    try:
        # ステップ1: 各スライドを個別に中間動画ファイルとして書き出す
        print("\n=== 中間クリップの生成 ===")
        for i, (media_path, video_clip, audio_path) in enumerate(zip(media_paths, video_clips, audio_paths)):
            print(f"  スライド {i+1} を処理中...")

            slide_clip = None

            if video_clip is not None:
                if audio_path and os.path.exists(audio_path):
                    audio_clip = AudioFileClip(audio_path)
                    if video_clip.duration < audio_clip.duration:
                        slide_clip = video_clip.fx(vfx.loop, duration=audio_clip.duration)
                        slide_clip = slide_clip.set_audio(audio_clip)
                    else:
                        slide_clip = video_clip.subclip(0, audio_clip.duration).set_audio(audio_clip)
                else:
                    slide_clip = video_clip

            elif media_path is not None:
                image_clip = ImageClip(media_path)
                if audio_path and os.path.exists(audio_path):
                    audio_clip = AudioFileClip(audio_path)
                    slide_clip = image_clip.set_duration(audio_clip.duration).set_audio(audio_clip)
                else:
                    slide_clip = image_clip.set_duration(default_duration)

            if slide_clip:
                intermediate_path = os.path.join(temp_dir, f"clip_{i+1:03d}.mp4")
                slide_clip.write_videofile(
                    intermediate_path, fps=30, codec="libx264",
                    preset="ultrafast", logger=None
                )
                intermediate_clip_paths.append(intermediate_path)
                print(f"    ✅ 中間ファイルを生成: {intermediate_path} ({slide_clip.duration:.1f}秒)")

            if video_clip: video_clip.close()
            if 'audio_clip' in locals() and audio_clip: audio_clip.close()
            if slide_clip: slide_clip.close()

        # ステップ2: 中間ファイルを高速に結合する
        print("\n=== 最終動画の結合 ===")
        if intermediate_clip_paths:
            # ★★★ 修正箇所 ★★★
            # VideoFileClipの読み込み時に verbose=False を指定してログを抑制
            clips_to_concatenate = [VideoFileClip(path, verbose=False) for path in intermediate_clip_paths]

            final_video = concatenate_videoclips(clips_to_concatenate, method="chain")

            print(f"動画を結合して出力中: {output_path}")
            final_video.write_videofile(
                output_path,
                fps=30,
                codec='libx264',
                audio_codec='aac',
                threads=os.cpu_count(),
                preset='ultrafast'
            )
            final_video.close()
        else:
            print("⚠️ 動画クリップがありませんでした。")
            return None

    finally:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
            print(f"\n🧹 一時ディレクトリを削除しました: {temp_dir}")

    print(f"\n🎉 完了: {output_path}")
    return output_path

## 10. メイン処理（通常モードと検証モード）

In [ ]:
# ============================================================================
# 10. メイン処理（通常モードと検証モード）- PDF対応版
# ============================================================================

def main(validation_mode: bool = False, use_existing_audio: bool = False, audio_dir: str = None):
    """
    メイン処理（HTML/PDF対応）

    Args:
        validation_mode: True の場合、音声生成をスキップ（検証用）
        use_existing_audio: True の場合、既存の音声ファイルを使用
        audio_dir: 既存音声ファイルのディレクトリ（use_existing_audio=Trueの場合）
    """
    mode_text = "検証モード" if validation_mode else ("既存音声使用モード" if use_existing_audio else "通常モード")
    print(f"=== スライド動画作成システム ({mode_text}) ===")
    print("Google Colab用 - HTML/PDF対応版")
    print()

    temp_html = None

    try:
        # 1. スライドファイル（HTML/PDF）とMarkdownファイルのアップロード
        slide_path, md_path, file_type = upload_main_files()

        # 2. ファイルタイプに応じて処理を分岐
        if file_type == 'pdf':
            print("\n=== PDFモード ===")

            # PDFからスライド画像を抽出
            image_paths = extract_slides_from_pdf(slide_path, OUTPUT_DIR)

            # 画像パスをmedia_pathsとvideo_clipsの形式に変換
            media_paths, video_clips = pdf_slides_to_video_clips(image_paths)

            animation_slides = {}  # PDFにはアニメーションなし

        else:  # HTML
            print("\n=== HTMLモード ===")

            # HTMLファイルを解析してリソースを特定
            print("\n=== HTMLファイル解析 ===")
            resources = analyze_html_resources(slide_path)
            total_resources = sum(len(files) for files in resources.values())

            if total_resources > 0:
                print(f"✅ {total_resources}個のリソースファイルが必要です")

                # リソースファイルのアップロード
                file_mapping = upload_required_resources(resources)

                # リソースパスを更新した一時HTMLファイルを作成
                print("\n=== 一時HTMLファイル作成 ===")
                temp_html = create_temp_html_with_resources(slide_path, file_mapping)
                print(f"✅ 一時HTMLファイル作成完了: {temp_html}")

                # アニメーションスライドの検出
                print("\n=== アニメーションスライドの検出 ===")
                animation_slides = detect_animation_slides(slide_path, file_mapping)
            else:
                print("✅ 追加リソースは必要ありません")
                temp_html = slide_path
                animation_slides = detect_animation_slides(slide_path)

            if animation_slides:
                print(f"✅ アニメーションを検出: スライド {[i+1 for i in animation_slides.keys()]}")
                for slide_num, duration in animation_slides.items():
                    print(f"   スライド {slide_num + 1}: {duration}秒")
            else:
                print("アニメーションスライドは検出されませんでした")

            # HTMLから画像/動画を生成
            print(f"\n=== ステップ2: HTML→画像/動画変換 ===")
            media_paths, video_clips = html_to_slides(temp_html, OUTPUT_DIR, animation_slides)

        # 3. 出力ディレクトリ準備
        setup_output_directory()

        # 4. 口述原稿を抽出
        print("\n=== ステップ1: 口述原稿抽出 ===")
        scripts = extract_speech_scripts(md_path)
        print(f"✅ 抽出完了: {len(scripts)}個の口述原稿")

        # 抽出した原稿を確認表示
        print("\n📝 抽出された口述原稿（プレビュー）:")
        for i, script in enumerate(scripts):
            print(f"\n--- スライド {i+1} ---")
            preview_text = script[:150] + "..." if len(script) > 150 else script
            print(preview_text)

        # 5. 音声ファイルを生成
        audio_paths = []
        if use_existing_audio:
            # 既存音声を使用
            print(f"\n=== ステップ3: 既存音声ファイル読み込み ===")
            if audio_dir is None:
                audio_dir = f"{OUTPUT_DIR}/audio"

            print(f"音声ファイルディレクトリ: {audio_dir}")
            for i in range(len(media_paths)):
                audio_file = f"{audio_dir}/slide_{i+1:03d}.mp3"
                if os.path.exists(audio_file):
                    audio_paths.append(audio_file)
                    print(f"✅ スライド {i+1}: {audio_file}")
                else:
                    print(f"⚠️  スライド {i+1}: 音声ファイルが見つかりません")
                    audio_paths.append(None)

        elif not validation_mode:
            # 新規に音声生成
            client = setup_gemini_client()
            print(f"\n=== ステップ3: 音声合成 ===")
            audio_paths = generate_audio_files(client, scripts, OUTPUT_DIR, prompts)
        else:
            # 検証モード（音声なし）
            print(f"\n=== ステップ3: 音声合成（スキップ - 検証モード） ===")
            audio_paths = [None] * len(media_paths)
            print("✅ 検証モードのため音声生成をスキップしました")

        # メディアと音声の数を調整
        max_slides = max(len(media_paths), len(audio_paths))
        while len(media_paths) < max_slides:
            media_paths.append(media_paths[-1] if media_paths else None)
            video_clips.append(video_clips[-1] if video_clips else None)
        while len(audio_paths) < max_slides:
            audio_paths.append(None)

        # 6. 動画を作成
        print(f"\n=== ステップ4: 動画作成 ===")
        if use_existing_audio:
            output_video_path = f"{OUTPUT_DIR}/presentation_video_with_existing_audio.mp4"
        else:
            output_video_path = f"{OUTPUT_DIR}/presentation_video{'_validation' if validation_mode else ''}.mp4"

        create_mixed_slide_video(media_paths, video_clips, audio_paths, output_video_path)

        # 7. 結果表示
        print("\n" + "="*50)
        print(f"🎉 処理完了！（{mode_text}）")
        print("="*50)
        print(f"📹 出力動画: {output_video_path}")
        print(f"🖼️  メディアファイル: {OUTPUT_DIR}/")
        if file_type == 'pdf':
            print(f"📄 PDFスライド画像: {OUTPUT_DIR}/pdf_slides/")
        if not validation_mode and not use_existing_audio:
            print(f"🎵 音声ファイル: {OUTPUT_DIR}/audio/")
        elif use_existing_audio:
            print(f"🎵 使用した音声ファイル: {audio_dir}/")

        # ファイルサイズとメタ情報を表示
        if os.path.exists(output_video_path):
            size_mb = os.path.getsize(output_video_path) / (1024 * 1024)
            print(f"📊 動画サイズ: {size_mb:.1f} MB")

            if file_type == 'html' and animation_slides:
                print(f"🎬 アニメーションスライド: {len(animation_slides)}個")

        print(f"\n💡 ファイルをダウンロードするには:")
        print(f"   files.download('{output_video_path}')")

        return output_video_path

    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        import traceback
        traceback.print_exc()
        return None

    finally:
        # クリーンアップ
        if temp_html and temp_html != slide_path and os.path.exists(temp_html):
            os.remove(temp_html)
            print(f"\n🧹 一時ファイルを削除しました: {temp_html}")

        # アップロードされたリソースの一時ディレクトリも削除
        temp_dir = Path("uploaded_resources")
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
            print(f"🧹 一時リソースディレクトリを削除しました")


In [ ]:
# ============================================================================
# 10. メイン処理（通常モード、検証モード、既存音声使用モード）
# ============================================================================

def get_existing_audio_files(audio_dir: str, num_slides: int) -> List[str]:
    """
    既存の音声ファイルを取得

    Args:
        audio_dir: 音声ファイルが格納されているディレクトリ
        num_slides: スライド数

    Returns:
        音声ファイルパスのリスト
    """
    audio_paths = []
    for i in range(num_slides):
        # 想定される音声ファイル名のパターン
        patterns = [
            f"{audio_dir}/script_{i+1:02d}.mp3",
            f"{audio_dir}/script_{i+1:02d}.wav",
        ]

        audio_file = None
        for pattern in patterns:
            if os.path.exists(pattern):
                audio_file = pattern
                break

        if audio_file:
            audio_paths.append(audio_file)
            print(f"✅ スライド {i+1}: {os.path.basename(audio_file)}")
        else:
            print(f"⚠️  スライド {i+1}: 音声ファイルが見つかりません")
            audio_paths.append(None)

    return audio_paths


def main(validation_mode: bool = False, use_existing_audio: bool = False, audio_dir: str = None):
    """
    メイン処理

    Args:
        validation_mode: True の場合、音声生成をスキップ（検証用）
        use_existing_audio: True の場合、既存の音声ファイルを使用
        audio_dir: 既存音声ファイルのディレクトリ（use_existing_audio=True時に必須）
    """
    # モード判定
    if validation_mode and use_existing_audio:
        raise ValueError("validation_mode と use_existing_audio は同時に指定できません")

    if use_existing_audio and not audio_dir:
        audio_dir = f"{OUTPUT_DIR}/audio"  # デフォルトの音声ディレクトリ

    # モード名の設定
    if validation_mode:
        mode_text = "検証モード"
    elif use_existing_audio:
        mode_text = "既存音声使用モード"
    else:
        mode_text = "通常モード"

    print(f"=== スライド動画作成システム ({mode_text}) ===")
    print("Google Colab用 - ファイルアップロード対応版")
    if use_existing_audio:
        print(f"音声ディレクトリ: {audio_dir}")
    print()

    temp_html = None

    try:
        # 1. HTMLとMarkdownファイルのアップロード
        slide_path, md_path, file_type = upload_main_files()

        # 2. ファイルタイプに応じて処理を分岐
        if file_type == 'pdf':
            print("\n=== PDFモード ===")

            # PDFからスライド画像を抽出
            image_paths = extract_slides_from_pdf(slide_path, OUTPUT_DIR)

            # 画像パスをmedia_pathsとvideo_clipsの形式に変換
            media_paths, video_clips = pdf_slides_to_video_clips(image_paths)

            animation_slides = {}  # PDFにはアニメーションなし

        else:  # HTML
            print("\n=== HTMLモード ===")
            # HTMLファイルを解析してリソースを特定
            print("\n=== HTMLファイル解析 ===")
            resources = analyze_html_resources(slide_path)
            total_resources = sum(len(files) for files in resources.values())

            if total_resources > 0:
                print(f"✅ {total_resources}個のリソースファイルが必要です")

                # 3. リソースファイルのアップロード
                file_mapping = upload_required_resources(resources)

                # 4. リソースパスを更新した一時HTMLファイルを作成
                print("\n=== 一時HTMLファイル作成 ===")
                temp_html = create_temp_html_with_resources(slide_path, file_mapping)
                print(f"✅ 一時HTMLファイル作成完了: {temp_html}")

                # 5. アニメーションスライドの検出
                print("\n=== アニメーションスライドの検出 ===")
                animation_slides = detect_animation_slides(slide_path, file_mapping)
            else:
                print("✅ 追加リソースは必要ありません")
                temp_html = slide_path
                animation_slides = detect_animation_slides(slide_path)

            if animation_slides:
                print(f"✅ アニメーションを検出: スライド {[i+1 for i in animation_slides.keys()]}")
                for slide_num, duration in animation_slides.items():
                    print(f"   スライド {slide_num + 1}: {duration}秒")
            else:
                print("アニメーションスライドは検出されませんでした")

            # 8. HTMLから画像/動画を生成
            print(f"\n=== ステップ2: HTML→画像/動画変換 ===")
            media_paths, video_clips = html_to_slides(temp_html, OUTPUT_DIR, animation_slides)

        # 6. 出力ディレクトリ準備
        setup_output_directory()

        # 7. 口述原稿を抽出
        print("\n=== ステップ1: 口述原稿抽出 ===")
        scripts = extract_speech_scripts(md_path)
        print(f"✅ 抽出完了: {len(scripts)}個の口述原稿")

        # 抽出した原稿を確認表示
        print("\n📝 抽出された口述原稿（プレビュー）:")
        for i, script in enumerate(scripts):
            print(f"\n--- スライド {i+1} ---")
            preview_text = script[:150] + "..." if len(script) > 150 else script
            print(preview_text)

        # 9. 音声ファイルを生成または取得
        audio_paths = []

        if validation_mode:
            # 検証モード：音声なし
            print(f"\n=== ステップ3: 音声合成（スキップ - 検証モード） ===")
            audio_paths = [None] * len(media_paths)
            print("✅ 検証モードのため音声生成をスキップしました")

        elif use_existing_audio:
            # 既存音声使用モード：既存ファイルを使用
            print(f"\n=== ステップ3: 音声ファイル読み込み（既存音声使用モード） ===")
            print(f"音声ファイルを検索中: {audio_dir}")

            # 既存の音声ファイルを確認
            if not os.path.exists(audio_dir):
                print(f"⚠️  警告: 音声ディレクトリが存在しません: {audio_dir}")
                print("音声なしで続行します")
                audio_paths = [None] * len(media_paths)
            else:
                audio_paths = get_existing_audio_files(audio_dir, len(media_paths))

                # 見つかった音声ファイル数を表示
                found_count = sum(1 for path in audio_paths if path is not None)
                print(f"\n✅ {found_count}/{len(media_paths)} 個の音声ファイルを検出")

                if found_count == 0:
                    print("⚠️  警告: 音声ファイルが1つも見つかりませんでした")
                elif found_count < len(media_paths):
                    print("⚠️  警告: 一部のスライドで音声ファイルが不足しています")

        else:
            # 通常モード：音声を生成
            # Gemini API設定
            client = setup_gemini_client()

            print(f"\n=== ステップ3: 音声合成 ===")
            audio_paths = generate_audio_files(client, scripts, OUTPUT_DIR, prompts)

        # メディアと音声の数を調整
        max_slides = max(len(media_paths), len(audio_paths))
        while len(media_paths) < max_slides:
            media_paths.append(media_paths[-1] if media_paths else None)
            video_clips.append(video_clips[-1] if video_clips else None)
        while len(audio_paths) < max_slides:
            audio_paths.append(None)

        # 10. 動画を作成
        print(f"\n=== ステップ4: 動画作成 ===")
        output_suffix = ''
        if validation_mode:
            output_suffix = '_validation'
        elif use_existing_audio:
            output_suffix = '_with_existing_audio'

        output_video_path = f"{OUTPUT_DIR}/presentation_video{output_suffix}.mp4"
        create_mixed_slide_video(media_paths, video_clips, audio_paths, output_video_path)

        # 11. 結果表示
        print("\n" + "="*50)
        print(f"🎉 処理完了！（{mode_text}）")
        print("="*50)
        print(f"📹 出力動画: {output_video_path}")
        print(f"🖼️  メディアファイル: {OUTPUT_DIR}/")

        if not validation_mode:
            if use_existing_audio:
                print(f"🎵 使用した音声ファイル: {audio_dir}/")
            else:
                print(f"🎵 生成した音声ファイル: {OUTPUT_DIR}/audio/")

        # ファイルサイズとメタ情報を表示
        if os.path.exists(output_video_path):
            size_mb = os.path.getsize(output_video_path) / (1024 * 1024)
            print(f"📊 動画サイズ: {size_mb:.1f} MB")

            if animation_slides:
                print(f"🎬 アニメーションスライド: {len(animation_slides)}個")

        print(f"\n💡 ファイルをダウンロードするには:")
        print(f"   files.download('{output_video_path}')")

        return output_video_path

    except Exception as e:
        print(f"❌ エラーが発生しました: {e}")
        import traceback
        traceback.print_exc()
        return None

    finally:
        # クリーンアップ
        if temp_html and temp_html != slide_path and os.path.exists(temp_html):
            os.remove(temp_html)
            print(f"\n🧹 一時ファイルを削除しました: {temp_html}")

        # アップロードされたリソースの一時ディレクトリも削除
        temp_dir = Path("uploaded_resources")
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
            print(f"🧹 一時リソースディレクトリを削除しました")

## 11. 便利な関数とユーティリティ

In [ ]:
# ============================================================================
# 11. 便利な関数とユーティリティ
# ============================================================================

def preview_scripts_from_file(md_path: str = None):
    """
    口述原稿をプレビュー表示
    """
    if md_path is None:
        print("Markdownファイルをアップロードしてください:")
        uploaded = files.upload()
        md_files = [f for f in uploaded.keys() if f.endswith('.md')]
        if not md_files:
            print("❌ Markdownファイルが見つかりません")
            return
        md_path = md_files[0]

    scripts = extract_speech_scripts(md_path)
    print(f"\n📝 抽出された口述原稿 ({len(scripts)}個):")
    for i, script in enumerate(scripts):
        print(f"\n=== スライド {i+1} ===")
        print(script)
        print("-" * 50)

def download_results():
    """生成されたファイルをダウンロード"""
    print("📥 ファイルダウンロード:")

    # 動画ファイル
    for video_name in ["presentation_video.mp4", "presentation_video_validation.mp4"]:
        video_path = f"{OUTPUT_DIR}/{video_name}"
        if os.path.exists(video_path):
            print(f"{video_name} をダウンロード中...")
            files.download(video_path)

    # ZIPファイルとして全ファイルをダウンロード
    import zipfile
    zip_path = f"{OUTPUT_DIR}/all_files.zip"

    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for root, dirs, filenames in os.walk(OUTPUT_DIR):
            for filename in filenames:
                if filename != 'all_files.zip':
                    file_path = os.path.join(root, filename)
                    arcname = os.path.relpath(file_path, OUTPUT_DIR)
                    zipf.write(file_path, arcname)

    print(f"全ファイルをZIPでダウンロード:")
    files.download(zip_path)

def test_svg_animation(svg_file_path):
    """SVGファイルのアニメーションをテスト"""
    has_animation, duration = check_svg_animation(svg_file_path)

    print(f"ファイル: {svg_file_path}")
    print(f"アニメーション: {'あり' if has_animation else 'なし'}")
    if has_animation:
        print(f"継続時間: {duration}秒")

    # SVGの内容を簡単に確認
    with open(svg_file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        if '<animate' in content:
            print("検出されたアニメーション要素:")
            animations = re.findall(r'<animate[^>]*>', content)
            for anim in animations[:3]:  # 最初の3つまで表示
                print(f"  {anim}")

def debug_animation_capture(html_path, slide_number=0):
    """特定のスライドのアニメーションをデバッグ"""
    driver = setup_chrome_driver()

    try:
        driver.get(f"file://{os.path.abspath(html_path)}")
        time.sleep(2)

        # 指定のスライドに移動
        for i in range(slide_number):
            driver.execute_script("document.dispatchEvent(new KeyboardEvent('keydown', {key: 'ArrowRight'}))")
            time.sleep(0.5)

        # アニメーションをトリガー
        trigger_svg_animations(driver)

        # スクリーンショットを撮って確認
        driver.save_screenshot(f"debug_slide_{slide_number}.png")
        print(f"デバッグ画像保存: debug_slide_{slide_number}.png")

        # 3秒後にもう一度スクリーンショット（アニメーション後）
        time.sleep(3)
        driver.save_screenshot(f"debug_slide_{slide_number}_after.png")
        print(f"アニメーション後画像保存: debug_slide_{slide_number}_after.png")

    finally:
        driver.quit()

## 12. 実行とインターフェース

In [ ]:
# ============================================================================
# 12. 実行インターフェース（PDF対応版）
# ============================================================================

def run_system(use_existing_audio=False, audio_dir=None):
    """
    通常モードで実行（音声あり）

    Args:
        use_existing_audio: 既存の音声ファイルを使用するか
        audio_dir: 既存音声ファイルのディレクトリパス
    """
    return main(validation_mode=False, use_existing_audio=use_existing_audio, audio_dir=audio_dir)

def run_validation():
    """
    検証モードで実行（音声なし・高速）
    """
    return main(validation_mode=True, use_existing_audio=False)


# 実行

In [ ]:
# ============================================================================
# 実行例（HTML/PDF対応）
# ============================================================================

# 1. 通常モード（HTML/PDF - 新規音声生成）
# run_system()

# 2. 検証モード（HTML/PDF - 音声なし・高速確認）
# run_validation()

# 3. 既存音声使用モード（HTML/PDF）
# run_system(use_existing_audio=True)

# 4. 既存音声使用モード（カスタムディレクトリ）
# run_system(use_existing_audio=True, audio_dir="/content/my_audio_files")

# 5. 直接main関数を呼び出す場合
# main(validation_mode=False, use_existing_audio=True, audio_dir="slide_output/audio")


In [ ]:
# 検証モードで実行（音声なし・高速）
#run_validation()

## 通常モード

In [ ]:
# 通常モードで実行（音声あり）
run_system()

## 既存音声使用モード

In [ ]:
# 既存音声使用モード（デフォルトディレクトリ）
main(use_existing_audio=True)

# 既存音声使用モード（カスタムディレクトリ）
# main(use_existing_audio=True, audio_dir="/content/my_audio_files")

# 通常モード（新規音声生成）
# main()

# 検証モード（音声なし）
# main(validation_mode=True)


In [ ]:
# デバッグ用：特定のスライドをテスト
#debug_animation_capture("temp_presentation.html", 6)  # 7番目のスライド

In [ ]:
# SVGファイルのアニメーションを確認
#test_svg_animation("fig__図1_エネルギー最小化.svg")

In [ ]:
# 口述原稿のプレビュー
# preview_scripts_from_file()

In [ ]:
# 結果ファイルのダウンロード
download_results()